# Week 5: Dynamic Mapping & Time-Machine Simulation

**Student Worksheet** — Fill in the code cells using AI assistance or your own code.

This week you will:
1. Call the CWA real-time rainfall API
2. Parse nested JSON into GeoDataFrame
3. Build interactive Folium maps with conditional styling
4. "Replay" Typhoon Fung-wong (2025) as a stress test
5. Overlay dynamic rainfall with shelter risk data

**Packages needed:** `geopandas`, `folium`, `requests`, `python-dotenv`, `branca`

## Cell [1]: Setup & Load Shelter Data

**What to do:**
- Import all required packages
- Load environment variables from .env
- Load shelter data from Week 3-4 (or create synthetic data)
- Print data summary

In [1]:
# Cell [1]: YOUR CODE HERE
# 1. Import packages: geopandas, pandas, numpy, folium, requests, json, os
# 2. Load .env with python-dotenv
# 3. Load Week 3-4 shelter data (use your ARIA v2 output, or create synthetic data)
# 4. Print shelter count, CRS, and columns

import geopandas as gpd
import pandas as pd
import numpy as np
import folium
import requests
import json
import os
from dotenv import load_dotenv
from shapely.geometry import Point
import matplotlib.pyplot as plt

# Load environment variables
load_dotenv('.env')
print("✅ Environment variables loaded")

# Load shelter data from CSV
shelters_df = pd.read_csv('避難收容處所點位檔案v9 (1).csv')
print(f"✅ Loaded {len(shelters_df)} shelters from CSV")

# Filter for Hualien County (花蓮縣)
hualien_shelters = shelters_df[shelters_df['縣市及鄉鎮市區'].str.contains('花蓮', na=False)].copy()
print(f"✅ Filtered to {len(hualien_shelters)} Hualien shelters")

# Create GeoDataFrame
geometry = [Point(lon, lat) for lon, lat in zip(hualien_shelters['經度'], hualien_shelters['緯度'])]
gdf_shelters = gpd.GeoDataFrame(
    hualien_shelters, 
    geometry=geometry, 
    crs='EPSG:4326'
)

# Convert to projected CRS for analysis
gdf_shelters = gdf_shelters.to_crs('EPSG:3826')

# Add synthetic risk data for Week 5 exercise
np.random.seed(42)
gdf_shelters['mean_elevation'] = np.random.uniform(50, 800, len(gdf_shelters))
gdf_shelters['max_slope'] = np.random.uniform(5, 45, len(gdf_shelters))
gdf_shelters['terrain_risk'] = np.where(gdf_shelters['max_slope'] > 25, 'HIGH', 
                                       np.where(gdf_shelters['max_slope'] > 15, 'MEDIUM', 'LOW'))

# Add shelter ID for tracking
gdf_shelters['shelter_id'] = [f'SH{i:03d}' for i in range(len(gdf_shelters))]

print(f"✅ Created shelter GeoDataFrame with {len(gdf_shelters)} shelters")
print(f"✅ CRS: {gdf_shelters.crs}")
print(f"✅ Columns: {list(gdf_shelters.columns)}")
print(f"✅ Terrain risk distribution:")
print(gdf_shelters['terrain_risk'].value_counts())

✅ Environment variables loaded
✅ Loaded 5973 shelters from CSV
✅ Filtered to 198 Hualien shelters
✅ Created shelter GeoDataFrame with 198 shelters
✅ CRS: EPSG:3826
✅ Columns: ['序號', '縣市及鄉鎮市區', '村里', '避難收容處所地址', '經度', '緯度', '避難收容處所名稱', '預計收容村里', '預計收容人數', '適用災害類別', '管理人姓名', '管理人電話', '室內', '室外', '適合避難弱者安置', 'geometry', 'mean_elevation', 'max_slope', 'terrain_risk', 'shelter_id']
✅ Terrain risk distribution:
terrain_risk
HIGH      108
LOW        49
MEDIUM     41
Name: count, dtype: int64


## Cell [2]: Fetch CWA Rainfall API

**What to do:**
- Write a function `fetch_cwa_api(api_key)` that calls the CWA rainfall endpoint
- Handle errors gracefully
- Return JSON response

In [2]:
# Cell [2]: YOUR CODE HERE
# Write a function fetch_cwa_api(api_key) that:
# 1. Calls https://opendata.cwa.gov.tw/api/v1/rest/datastore/O-A0002-001
# 2. Returns the JSON response
# 3. Handles errors with try/except

def fetch_cwa_api(api_key):
    """
    Fetch real-time rainfall data from CWA API
    
    Args:
        api_key (str): CWA API authorization key
        
    Returns:
        dict: JSON response from API or None if error
    """
    url = "https://opendata.cwa.gov.tw/api/v1/rest/datastore/O-A0002-001"
    headers = {
        "Authorization": api_key,
        "Content-Type": "application/json"
    }
    
    try:
        print("🌧️ Fetching real-time rainfall data from CWA API...")
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        
        data = response.json()
        print(f"✅ Successfully fetched data from CWA API")
        return data
        
    except requests.exceptions.RequestException as e:
        print(f"❌ Error fetching CWA API: {e}")
        return None
    except json.JSONDecodeError as e:
        print(f"❌ Error parsing JSON response: {e}")
        return None
    except Exception as e:
        print(f"❌ Unexpected error: {e}")
        return None

# Test the function (commented out to avoid real API calls during testing)
# api_key = os.getenv('CWA_API_KEY')
# if api_key:
#     test_data = fetch_cwa_api(api_key)
#     if test_data:
#         print(f"✅ Test successful - got {len(test_data.get('records', {}).get('Station', []))} stations")
# else:
#     print("❌ No CWA_API_KEY found in environment")

print("✅ fetch_cwa_api function defined")

✅ fetch_cwa_api function defined


## Cell [3]: Parse Rainfall JSON → GeoDataFrame

**What to do:**
- Extract station data from nested JSON structure
- Filter out invalid data (-998 values)
- Create GeoDataFrame with proper CRS

In [3]:
# Cell [3]: YOUR CODE HERE
# Write a function parse_rainfall_json(data) that:
# 1. Detects JSON format (CWA API vs CoLife-converted JSON vs XML download)
# 2. Extracts station list from the correct root path
# 3. Gets: StationName, StationId, lat, lon, rain_1hr, rain_3hr, rain_24hr
# 4. Filters out -998 values (NoData sentinel)
# 5. Returns a GeoDataFrame with CRS EPSG:4326

def normalize_cwa_json(raw):
    """
    Normalize different CWA JSON formats to standard station list
    
    Args:
        raw (dict): Raw JSON data from different sources
        
    Returns:
        list: Standardized station list
    """
    if 'records' in raw and 'Station' in raw['records']:
        # CWA API format or CoLife converted format
        return raw['records']['Station']
    elif 'cwaopendata' in raw and 'dataset' in raw['cwaopendata']:
        # CWA XML download format
        return raw['cwaopendata']['dataset']['Station']
    else:
        raise ValueError("Unknown JSON format - cannot find station data")

def parse_rainfall_json(data):
    """
    Parse rainfall JSON data into GeoDataFrame
    
    Args:
        data (dict): JSON data from CWA API or CoLife
        
    Returns:
        GeoDataFrame: Rainfall stations with CRS EPSG:4326
    """
    try:
        # Normalize station list
        stations = normalize_cwa_json(data)
        print(f"✅ Found {len(stations)} stations in data")
        
        parsed_stations = []
        
        for station in stations:
            try:
                # Extract basic station info
                station_name = station.get('StationName', 'Unknown')
                station_id = station.get('StationId', 'Unknown')
                
                # Extract coordinates (handle different formats)
                coordinates = station.get('GeoInfo', {}).get('Coordinates', [])
                if not coordinates:
                    continue
                    
                # Handle coordinate format differences
                if len(coordinates) >= 2:
                    # CWA API format - pick WGS84 (usually the second one)
                    coord = coordinates[1] if coordinates[1].get('CoordinateName') == 'WGS84' else coordinates[0]
                else:
                    # CoLife format - only one coordinate set (WGS84)
                    coord = coordinates[0]
                
                lat = coord.get('StationLatitude')
                lon = coord.get('StationLongitude')
                
                if lat is None or lon is None:
                    continue
                
                # Extract rainfall data
                rainfall_element = station.get('RainfallElement', {})
                
                # Get precipitation values (handle both string and number formats)
                past_1hr = rainfall_element.get('Past1hr', {}).get('Precipitation', 0)
                past_3hr = rainfall_element.get('Past3hr', {}).get('Precipitation', 0)
                past_24hr = rainfall_element.get('Past24hr', {}).get('Precipitation', 0)
                
                # Convert to float and filter -998 values
                try:
                    rain_1hr = float(past_1hr) if past_1hr != -998 else 0
                    rain_3hr = float(past_3hr) if past_3hr != -998 else 0
                    rain_24hr = float(past_24hr) if past_24hr != -998 else 0
                except (ValueError, TypeError):
                    rain_1hr = rain_3hr = rain_24hr = 0
                
                # Get location info
                county_name = station.get('GeoInfo', {}).get('CountyName', 'Unknown')
                town_name = station.get('GeoInfo', {}).get('TownName', 'Unknown')
                
                parsed_stations.append({
                    'station_name': station_name,
                    'station_id': station_id,
                    'latitude': lat,
                    'longitude': lon,
                    'county': county_name,
                    'town': town_name,
                    'rain_1hr': rain_1hr,
                    'rain_3hr': rain_3hr,
                    'rain_24hr': rain_24hr
                })
                
            except Exception as e:
                print(f"⚠️ Error parsing station {station.get('StationName', 'Unknown')}: {e}")
                continue
        
        # Create GeoDataFrame
        if not parsed_stations:
            raise ValueError("No valid stations found in data")
        
        gdf = gpd.GeoDataFrame(
            parsed_stations,
            geometry=gpd.points_from_xy([s['longitude'] for s in parsed_stations], 
                                     [s['latitude'] for s in parsed_stations]),
            crs='EPSG:4326'
        )
        
        print(f"✅ Successfully parsed {len(gdf)} rainfall stations")
        print(f"✅ Rainfall statistics - 1hr: {gdf['rain_1hr'].max():.1f}mm max, {gdf['rain_1hr'].mean():.1f}mm mean")
        
        return gdf
        
    except Exception as e:
        print(f"❌ Error parsing rainfall JSON: {e}")
        raise

print("✅ parse_rainfall_json function defined")

✅ parse_rainfall_json function defined


## Cell [4]: Mode Switcher (LIVE vs SIMULATION)

**What to do:**
- Read `APP_MODE` from .env
- If LIVE: fetch real rainfall data from API
- If SIMULATION: load fallback JSON file
- Parse using the same function in both cases

In [4]:
# Cell [4]: YOUR CODE HERE
# Mode Switcher:
# 1. Read APP_MODE from .env (default: 'SIMULATION')
# 2. If LIVE: call fetch_cwa_api() → parse_rainfall_json()
# 3. If SIMULATION: load fungwong_202511.json → parse_rainfall_json()
# 4. KEY INSIGHT: same parse function for both!

# Read application mode
app_mode = os.getenv('APP_MODE', 'SIMULATION')
print(f"🎯 Application Mode: {app_mode}")

gdf_rainfall = None

if app_mode == 'LIVE':
    # LIVE mode - fetch real-time data
    api_key = os.getenv('CWA_API_KEY')
    if not api_key:
        print("❌ No CWA_API_KEY found in .env - switching to SIMULATION mode")
        app_mode = 'SIMULATION'
    else:
        raw_data = fetch_cwa_api(api_key)
        if raw_data:
            try:
                gdf_rainfall = parse_rainfall_json(raw_data)
            except Exception as e:
                print(f"❌ Failed to parse live data: {e}")
                print("🔄 Switching to SIMULATION mode")
                app_mode = 'SIMULATION'
        else:
            print("❌ Failed to fetch live data - switching to SIMULATION mode")
            app_mode = 'SIMULATION'

if app_mode == 'SIMULATION' and gdf_rainfall is None:
    # SIMULATION mode - load Typhoon Fung-wong data
    simulation_file = os.getenv('SIMULATION_DATA', 'fungwong_202511.json')
    print(f"🌀 Loading simulation data from: {simulation_file}")
    
    try:
        with open(simulation_file, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)
        gdf_rainfall = parse_rainfall_json(raw_data)
        print("✅ Typhoon Fung-wong simulation data loaded successfully")
    except FileNotFoundError:
        print(f"❌ Simulation file not found: {simulation_file}")
        raise
    except Exception as e:
        print(f"❌ Error loading simulation data: {e}")
        raise

# Display summary
if gdf_rainfall is not None:
    print(f"✅ Successfully loaded rainfall data in {app_mode} mode")
    print(f"📊 Total stations: {len(gdf_rainfall)}")
    print(f"📍 CRS: {gdf_rainfall.crs}")
    
    # Show high rainfall stations
    high_rain = gdf_rainfall[gdf_rainfall['rain_1hr'] > 40]
    if len(high_rain) > 0:
        print(f"⚠️ High rainfall stations (>40mm/hr): {len(high_rain)}")
        print(high_rain[['station_name', 'rain_1hr', 'county']].nlargest(5, 'rain_1hr'))
    else:
        print("🌤️ No high rainfall stations (>40mm/hr) found")
        
else:
    print("❌ Failed to load rainfall data")

print(f"✅ Mode switcher completed - running in {app_mode} mode")

🎯 Application Mode: SIMULATION
🌀 Loading simulation data from: fungwong_202511.json
✅ Found 1256 stations in data
✅ Successfully parsed 1256 rainfall stations
✅ Rainfall statistics - 1hr: 130.5mm max, 1.6mm mean
✅ Typhoon Fung-wong simulation data loaded successfully
✅ Successfully loaded rainfall data in SIMULATION mode
📊 Total stations: 1256
📍 CRS: EPSG:4326
⚠️ High rainfall stations (>40mm/hr): 7
     station_name  rain_1hr county
250            蘇澳     130.5    宜蘭縣
117       國五S047K      80.5    宜蘭縣
610            五結      71.0    宜蘭縣
1038           冬山      61.5    宜蘭縣
118       國五S041K      46.0    宜蘭縣
✅ Mode switcher completed - running in SIMULATION mode


## Cell [5]: Create Base Folium Map

**What to do:**
- Create a Folium map centered on Hualien County
- Use OpenStreetMap or Satellite basemap
- Set initial zoom level

In [5]:
# Cell [5]: YOUR CODE HERE
# Create a base Folium map:
# 1. Center on Hualien (latitude ~23.98, longitude ~121.55)
# 2. Use tiles='OpenStreetMap' or tiles='Satellite'
# 3. Set zoom_start=10
# 4. Assign to variable `m`

# Get map center from environment or use default
map_center_lat = float(os.getenv('MAP_CENTER_LAT', 23.98))
map_center_lon = float(os.getenv('MAP_CENTER_LON', 121.55))
map_zoom = int(os.getenv('MAP_ZOOM_START', 10))

# Create base Folium map
m = folium.Map(
    location=[map_center_lat, map_center_lon],
    zoom_start=map_zoom,
    tiles='OpenStreetMap'
)

print(f"🗺️ Created Folium map centered at [{map_center_lat}, {map_center_lon}] with zoom {map_zoom}")
print("✅ Base map ready for adding layers")

🗺️ Created Folium map centered at [23.98, 121.55] with zoom 10
✅ Base map ready for adding layers


## Cell [6]: Add Rainfall CircleMarkers with Conditional Styling

**What to do:**
- Write a function `rain_color(rain_value)` that returns color based on rainfall amount
- Add CircleMarker for each rainfall station
- Size and color represent rainfall intensity

In [6]:
# Cell [6]: YOUR CODE HERE
# 1. Write function rain_color(rain_mm) that returns:
#    - 'green'  if rain_mm < 10 mm/hr    (safe)
#    - 'gold'   if 10 <= rain_mm < 40    (caution)
#    - 'orange' if 40 <= rain_mm < 80    (warning)
#    - 'red'    if rain_mm >= 80 mm/hr   (danger)
# 2. Loop through gdf_rainfall and add CircleMarker for each station
# 3. Radius proportional to rain_1hr: radius = max(5, rain_mm / 5)

def rain_color(rain_mm):
    """Return color based on rainfall intensity"""
    if rain_mm < 10:
        return 'green'
    elif rain_mm < 40:
        return 'gold'
    elif rain_mm < 80:
        return 'orange'
    else:
        return 'red'

def rain_radius(rain_mm):
    """Return marker radius based on rainfall intensity"""
    return max(5, rain_mm / 5)

# Add rainfall CircleMarkers to map
print("🌧️ Adding rainfall stations to map...")

rainfall_layer = folium.FeatureGroup(name="Rainfall Stations")

for idx, station in gdf_rainfall.iterrows():
    rain_1hr = station['rain_1hr']
    
    # Create popup with station information
    popup_text = f"""
    <b>{station['station_name']}</b><br>
    County: {station['county']}<br>
    Town: {station['town']}<br>
    1-hr Rain: {rain_1hr:.1f} mm<br>
    3-hr Rain: {station['rain_3hr']:.1f} mm<br>
    24-hr Rain: {station['rain_24hr']:.1f} mm
    """
    
    # Add CircleMarker
    folium.CircleMarker(
        location=[station['latitude'], station['longitude']],
        radius=rain_radius(rain_1hr),
        popup=folium.Popup(popup_text, max_width=200),
        tooltip=f"{station['station_name']}: {rain_1hr:.1f}mm/hr",
        color='black',
        weight=1,
        fillColor=rain_color(rain_1hr),
        fillOpacity=0.7
    ).add_to(rainfall_layer)

# Add the rainfall layer to the map
rainfall_layer.add_to(m)

print(f"✅ Added {len(gdf_rainfall)} rainfall stations to map")

# Add legend for rainfall colors
legend_html = '''
<div style="position: fixed; 
     top: 10px; right: 10px; width: 150px; height: 120px; 
     border:2px solid grey; z-index:9999; font-size:14px;
     background-color:white; padding: 10px">
<b>Rainfall (mm/hr)</b><br>
<span style="color:green">●</span> < 10 (Safe)<br>
<span style="color:gold">●</span> 10-40 (Caution)<br>
<span style="color:orange">●</span> 40-80 (Warning)<br>
<span style="color:red">●</span> ≥ 80 (Danger)
</div>
'''

m.get_root().html.add_child(folium.Element(legend_html))
print("✅ Added rainfall legend")

🌧️ Adding rainfall stations to map...
✅ Added 1256 rainfall stations to map
✅ Added rainfall legend


## Cell [7]: Add HeatMap Layer

**What to do:**
- Import HeatMap from folium.plugins
- Create a heat layer showing rainfall intensity
- Add to folium map

In [7]:
# Cell [7]: YOUR CODE HERE
# 1. Import HeatMap from folium.plugins
# 2. Create list of [lat, lon, rain_1hr] for each station
# 3. Add HeatMap(data, name='Rainfall Heatmap', show=False) to map m

from folium.plugins import HeatMap

# Create heat data for HeatMap
print("🔥 Creating rainfall heatmap...")

heat_data = []
for idx, station in gdf_rainfall.iterrows():
    lat = station['latitude']
    lon = station['longitude']
    rain_1hr = station['rain_1hr']
    
    # Only include stations with measurable rainfall
    if rain_1hr > 0:
        heat_data.append([lat, lon, rain_1hr])

# Add HeatMap layer
if heat_data:
    heatmap_layer = HeatMap(
        heat_data,
        name='Rainfall Heatmap',
        show=False,
        radius=15,
        blur=10,
        gradient={
            0.0: 'green',
            0.3: 'gold', 
            0.6: 'orange',
            1.0: 'red'
        }
    )
    heatmap_layer.add_to(m)
    print(f"✅ Added HeatMap with {len(heat_data)} data points")
else:
    print("⚠️ No rainfall data for HeatMap")

print("✅ HeatMap layer created and added to map")

🔥 Creating rainfall heatmap...

✅ Added HeatMap with 315 data points
✅ HeatMap layer created and added to map


## Cell [8]: Add LayerControl

**What to do:**
- Enable layer visibility toggle for CircleMarkers and HeatMap
- Add LayerControl to map

In [8]:
# Cell [8]: YOUR CODE HERE
# 1. Import LayerControl from folium
# 2. Add LayerControl(collapsed=False) to map m
# 3. This lets users toggle layers on/off

from folium import LayerControl

# Add LayerControl to map
layer_control = LayerControl(collapsed=False)
m.add_child(layer_control)

print("✅ LayerControl added to map - users can now toggle layers")

# Display the map
print("🗺️ Displaying interactive rainfall map...")
display(m)

✅ LayerControl added to map - users can now toggle layers
🗺️ Displaying interactive rainfall map...


## Cell [9]: Add Shelter Risk Popups

**What to do:**
- Add shelter locations to map
- Color-code by risk_level
- Include rich popup with shelter name and risk info

In [9]:
# Cell [9]: YOUR CODE HERE
# 1. Loop through gdf_shelters and add Marker for each shelter
# 2. Color by risk_level: 'low'→blue, 'medium'→orange, 'high'→red
# 3. Create rich popup with HTML:
#    - Shelter name
#    - Risk level
#    - Terrain risk
#    - Mean elevation
#    - Max slope

def shelter_icon_color(risk_level):
    """Return icon color based on risk level"""
    if risk_level == 'HIGH':
        return 'red'
    elif risk_level == 'MEDIUM':
        return 'orange'
    else:
        return 'blue'

# Add shelter markers to map
print("🏠 Adding shelter markers to map...")

shelter_layer = folium.FeatureGroup(name="Shelters")

for idx, shelter in gdf_shelters.iterrows():
    # Get shelter info
    name = shelter['避難收容處所名稱']
    address = shelter['避難收容處所地址']
    terrain_risk = shelter['terrain_risk']
    elevation = shelter['mean_elevation']
    slope = shelter['max_slope']
    shelter_id = shelter['shelter_id']
    
    # Get coordinates for Folium (convert back to WGS84)
    geom = shelter.geometry
    lon, lat = geom.x, geom.y
    
    # Convert to WGS84 for display
    shelter_wgs84 = gpd.GeoDataFrame(
        {'shelter_id': [shelter_id], 'geometry': [geom]}, 
        crs='EPSG:3826'
    ).to_crs('EPSG:4326')
    
    display_lon, display_lat = shelter_wgs84.geometry[0].x, shelter_wgs84.geometry[0].y
    
    # Create popup HTML
    popup_html = f"""
    <div style="width: 200px">
        <h4>{name}</h4>
        <b>Address:</b> {address}<br>
        <b>Shelter ID:</b> {shelter_id}<br>
        <b>Terrain Risk:</b> {terrain_risk}<br>
        <b>Elevation:</b> {elevation:.1f}m<br>
        <b>Max Slope:</b> {slope:.1f}°<br>
        <b>Capacity:</b> {shelter.get('預計收容人數', 'N/A')} people<br>
        <b>Suitable for:</b> {shelter.get('適合避難弱者安置', 'No')}
    </div>
    """
    
    # Create marker
    folium.Marker(
        location=[display_lat, display_lon],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"{name} ({terrain_risk} risk)",
        icon=folium.Icon(
            color=shelter_icon_color(terrain_risk),
            icon='home',
            prefix='fa'
        )
    ).add_to(shelter_layer)

# Add shelter layer to map
shelter_layer.add_to(m)

print(f"✅ Added {len(gdf_shelters)} shelter markers to map")

# Add shelter legend
shelter_legend_html = '''
<div style="position: fixed; 
     top: 140px; right: 10px; width: 150px; height: 100px; 
     border:2px solid grey; z-index:9999; font-size:14px;
     background-color:white; padding: 10px">
<b>Shelter Risk</b><br>
<span style="color:blue">🏠</span> LOW Risk<br>
<span style="color:orange">🏠</span> MEDIUM Risk<br>
<span style="color:red">🏠</span> HIGH Risk
</div>
'''

m.get_root().html.add_child(folium.Element(shelter_legend_html))
print("✅ Added shelter legend")

🏠 Adding shelter markers to map...


✅ Added 198 shelter markers to map
✅ Added shelter legend


---

# Lab 1: CWA API → Folium Map (25 minutes)

**Goal**: Call the rainfall API (or load fallback), parse JSON, create an interactive Folium map.

> **Fallback**: If CWA API doesn't work, load `data/scenarios/fungwong_202511.json` instead. The structure is similar (both use `records.Station[]`) but has minor differences — your `parse_rainfall_json()` should handle both via `normalize_cwa_json()`.

**Checklist:**
- [ ] Rainfall data loaded (API or fallback)
- [ ] GeoDataFrame parsed with correct CRS
- [ ] Folium map created with CircleMarkers
- [ ] Map saved as HTML
- [ ] Can toggle layers on/off

### Lab 1 Step 1: Load Data (API or Fallback)

In [10]:
# Lab 1 Step 1: YOUR CODE HERE
# 1. Read API_KEY from .env
# 2. Try: fetch_cwa_api(api_key)
# 3. If error or None: load 'data/scenarios/fungwong_202511.json'
# 4. Parse JSON → gdf_rainfall
# 5. Print shape, columns, CRS

print("🌧️ Lab 1: Loading rainfall data (API or fallback)")

# Try API first if in LIVE mode
api_key = os.getenv('CWA_API_KEY')
use_api = os.getenv('APP_MODE', 'SIMULATION') == 'LIVE' and api_key

if use_api:
    print("📡 Attempting to fetch live data from CWA API...")
    raw_data = fetch_cwa_api(api_key)
    if raw_data:
        try:
            gdf_rainfall_lab = parse_rainfall_json(raw_data)
            print("✅ Successfully loaded live API data")
        except Exception as e:
            print(f"❌ Failed to parse API data: {e}")
            print("🔄 Falling back to simulation data")
            use_api = False
    else:
        print("❌ Failed to fetch API data")
        print("🔄 Falling back to simulation data")
        use_api = False

if not use_api:
    print("🌀 Loading Typhoon Fung-wong simulation data...")
    with open('fungwong_202511.json', 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    gdf_rainfall_lab = parse_rainfall_json(raw_data)
    print("✅ Successfully loaded simulation data")

# Display summary
print(f"📊 Rainfall data summary:")
print(f"   Shape: {gdf_rainfall_lab.shape}")
print(f"   Columns: {list(gdf_rainfall_lab.columns)}")
print(f"   CRS: {gdf_rainfall_lab.crs}")
print(f"   Data source: {'LIVE API' if use_api else 'SIMULATION'}")

🌧️ Lab 1: Loading rainfall data (API or fallback)
🌀 Loading Typhoon Fung-wong simulation data...
✅ Found 1256 stations in data
✅ Successfully parsed 1256 rainfall stations
✅ Rainfall statistics - 1hr: 130.5mm max, 1.6mm mean
✅ Successfully loaded simulation data
📊 Rainfall data summary:
   Shape: (1256, 10)
   Columns: ['station_name', 'station_id', 'latitude', 'longitude', 'county', 'town', 'rain_1hr', 'rain_3hr', 'rain_24hr', 'geometry']
   CRS: EPSG:4326
   Data source: SIMULATION


### Lab 1 Step 2: Parse JSON → GeoDataFrame

In [11]:
# Lab 1 Step 2: YOUR CODE HERE
# 1. Check gdf_rainfall has columns: rain_1hr, rain_3hr, rain_24hr
# 2. Check CRS is EPSG:4326
# 3. Display first 5 rows
# 4. Print statistics: min/max/mean rainfall

print("📊 Lab 1: Analyzing rainfall data structure")

# Check required columns
required_columns = ['rain_1hr', 'rain_3hr', 'rain_24hr']
missing_columns = [col for col in required_columns if col not in gdf_rainfall_lab.columns]

if missing_columns:
    print(f"❌ Missing columns: {missing_columns}")
else:
    print("✅ All required rainfall columns present")

# Check CRS
if str(gdf_rainfall_lab.crs) == 'EPSG:4326':
    print("✅ CRS is correctly set to EPSG:4326")
else:
    print(f"⚠️ CRS is {gdf_rainfall_lab.crs}, expected EPSG:4326")

# Display sample data
print("\n📋 Sample rainfall stations:")
display(gdf_rainfall_lab.head())

# Rainfall statistics
print("\n📈 Rainfall Statistics:")
print(f"   1-hour rainfall:")
print(f"      Min: {gdf_rainfall_lab['rain_1hr'].min():.1f} mm")
print(f"      Max: {gdf_rainfall_lab['rain_1hr'].max():.1f} mm")
print(f"      Mean: {gdf_rainfall_lab['rain_1hr'].mean():.1f} mm")
print(f"      Std: {gdf_rainfall_lab['rain_1hr'].std():.1f} mm")

print(f"   3-hour rainfall:")
print(f"      Min: {gdf_rainfall_lab['rain_3hr'].min():.1f} mm")
print(f"      Max: {gdf_rainfall_lab['rain_3hr'].max():.1f} mm")
print(f"      Mean: {gdf_rainfall_lab['rain_3hr'].mean():.1f} mm")

print(f"   24-hour rainfall:")
print(f"      Min: {gdf_rainfall_lab['rain_24hr'].min():.1f} mm")
print(f"      Max: {gdf_rainfall_lab['rain_24hr'].max():.1f} mm")
print(f"      Mean: {gdf_rainfall_lab['rain_24hr'].mean():.1f} mm")

# Find stations with highest rainfall
print("\n🌧️ Top 5 stations by 1-hour rainfall:")
top_stations = gdf_rainfall_lab.nlargest(5, 'rain_1hr')
display(top_stations[['station_name', 'county', 'rain_1hr', 'rain_3hr']])

📊 Lab 1: Analyzing rainfall data structure
✅ All required rainfall columns present
✅ CRS is correctly set to EPSG:4326

📋 Sample rainfall stations:


,station_name,station_id,latitude,longitude,county,town,rain_1hr,rain_3hr,rain_24hr,geometry
0,國一S072K,CAC010,24.895830,121.134200,桃園市,楊梅區,0.0,0.5,15.0,POINT (121.1342 24.89583)
1,馬光農場,C2K620,23.721453,120.378864,雲林縣,虎尾鎮,0.0,0.0,0.5,POINT (120.37886 23.72145)
2,出雲,C2FB50,24.206469,120.902794,臺中市,和平區,0.0,0.0,1.0,POINT (120.90279 24.20647)
3,頭櫃山,C2FB60,24.129842,120.858925,臺中市,新社區,0.0,0.0,0.5,POINT (120.85892 24.12984)
4,三隻寮,C2H9D0,24.093078,120.861578,南投縣,國姓鄉,0.0,0.0,1.0,POINT (120.86158 24.09308)



📈 Rainfall Statistics:
   1-hour rainfall:
      Min: 0.0 mm
      Max: 130.5 mm
      Mean: 1.6 mm
      Std: 6.7 mm
   3-hour rainfall:
      Min: 0.0 mm
      Max: 217.0 mm
      Mean: 5.1 mm
   24-hour rainfall:
      Min: 0.0 mm
      Max: 758.0 mm
      Mean: 49.3 mm

🌧️ Top 5 stations by 1-hour rainfall:


,station_name,county,rain_1hr,rain_3hr
250,蘇澳,宜蘭縣,130.5,198.5
117,國五S047K,宜蘭縣,80.5,150.5
610,五結,宜蘭縣,71.0,123.5
1038,冬山,宜蘭縣,61.5,217.0
118,國五S041K,宜蘭縣,46.0,79.0


### Lab 1 Step 3: Build Folium Map + CircleMarkers

In [12]:
# Lab 1 Step 3: YOUR CODE HERE
# 1. Create Folium map (reuse Cell [5] & [6])
# 2. Add CircleMarkers for rainfall stations
# 3. Add HeatMap layer
# 4. Add LayerControl
# 5. Display map

print("🗺️ Lab 1: Building interactive rainfall map")

# Create new map for Lab 1
m_lab1 = folium.Map(
    location=[23.98, 121.55],
    zoom_start=10,
    tiles='OpenStreetMap'
)

# Add rainfall CircleMarkers
print("🌧️ Adding rainfall stations...")
rainfall_layer_lab1 = folium.FeatureGroup(name="Rainfall Stations")

for idx, station in gdf_rainfall_lab.iterrows():
    rain_1hr = station['rain_1hr']
    
    folium.CircleMarker(
        location=[station['latitude'], station['longitude']],
        radius=max(5, rain_1hr / 5),
        popup=f"{station['station_name']}<br>1hr: {rain_1hr:.1f}mm<br>3hr: {station['rain_3hr']:.1f}mm",
        tooltip=f"{station['station_name']}: {rain_1hr:.1f}mm/hr",
        color='black',
        weight=1,
        fillColor=rain_color(rain_1hr),
        fillOpacity=0.7
    ).add_to(rainfall_layer_lab1)

rainfall_layer_lab1.add_to(m_lab1)

# Add HeatMap
print("🔥 Adding rainfall heatmap...")
heat_data_lab1 = [[row['latitude'], row['longitude'], row['rain_1hr']] 
                  for idx, row in gdf_rainfall_lab.iterrows() if row['rain_1hr'] > 0]

if heat_data_lab1:
    HeatMap(
        heat_data_lab1,
        name='Rainfall Heatmap',
        show=False,
        radius=15,
        blur=10
    ).add_to(m_lab1)

# Add LayerControl
LayerControl(collapsed=False).add_to(m_lab1)

print("✅ Lab 1 map completed")
display(m_lab1)

🗺️ Lab 1: Building interactive rainfall map
🌧️ Adding rainfall stations...
🔥 Adding rainfall heatmap...
✅ Lab 1 map completed


### Lab 1 Step 4: Save Map as HTML

In [13]:
# Lab 1 Step 4: YOUR CODE HERE
# 1. Save map to 'output/rainfall_map_week5.html'
# 2. Verify file was created
# 3. Print file size

print("💾 Lab 1: Saving rainfall map")

# Create output directory if it doesn't exist
output_dir = 'output'
os.makedirs(output_dir, exist_ok=True)

# Save the map
output_file = os.path.join(output_dir, 'rainfall_map_week5.html')
m_lab1.save(output_file)

# Verify file was created
if os.path.exists(output_file):
    file_size = os.path.getsize(output_file)
    print(f"✅ Map saved successfully: {output_file}")
    print(f"📁 File size: {file_size:,} bytes ({file_size/1024:.1f} KB)")
else:
    print(f"❌ Failed to save map: {output_file}")

print("✅ Lab 1 completed - rainfall map saved")

💾 Lab 1: Saving rainfall map


✅ Map saved successfully: output\rainfall_map_week5.html
📁 File size: 1,634,778 bytes (1596.5 KB)
✅ Lab 1 completed - rainfall map saved


---

# 🔬 Lab 2: Typhoon Fung-wong Simulation (15 minutes)

**Goal**: Switch to SIMULATION mode, overlay typhoon rainfall with shelter risk data.

> **Context**: It's 2025-11-11 14:00. Typhoon Fung-wong is hitting eastern Taiwan.
> Suao: 130.5mm/hr. Mataian Creek is forming a landslide dam.

**Checklist:**
- [ ] Simulation data loaded
- [ ] High-rainfall stations identified
- [ ] Shelters within 5km radius found
- [ ] Risk map created and saved

### Lab 2 Step 1: Load Simulation JSON + Filter High-Rain Stations

In [14]:
# Lab 2 Step 1: YOUR CODE HERE
# 1. Load 'data/scenarios/fungwong_202511.json'
# 2. Parse with parse_rainfall_json()
# 3. Filter: rain_1hr > 30 mm/hr (heavy rain)
# 4. Print how many high-rain stations found
# 5. Find station with max rain_1hr (Suao should be 130.5mm/hr)

print("🌀 Lab 2: Typhoon Fung-wong Simulation Analysis")

# Load Typhoon Fung-wong simulation data
print("🌀 Loading Typhoon Fung-wong simulation data...")
with open('fungwong_202511.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

gdf_fungwong = parse_rainfall_json(raw_data)

# Filter for heavy rain (>30mm/hr)
heavy_rain_threshold = 30
gdf_heavy_rain = gdf_fungwong[gdf_fungwong['rain_1hr'] > heavy_rain_threshold]

print(f"✅ Loaded {len(gdf_fungwong)} total stations")
print(f"⚠️ Found {len(gdf_heavy_rain)} stations with heavy rain (> {heavy_rain_threshold}mm/hr)")

# Find station with maximum rainfall
max_rain_station = gdf_fungwong.loc[gdf_fungwong['rain_1hr'].idxmax()]
max_rain_value = max_rain_station['rain_1hr']
max_rain_name = max_rain_station['station_name']
max_rain_county = max_rain_station['county']

print(f"🌧️ Station with maximum rainfall:")
print(f"   Name: {max_rain_name}")
print(f"   County: {max_rain_county}")
print(f"   1-hour rainfall: {max_rain_value:.1f} mm/hr")

# Check if it's Suao as expected
if '蘇澳' in max_rain_name:
    print("✅ Maximum rainfall station is Suao as expected!")
else:
    print(f"⚠️ Expected Suao, but found {max_rain_name}")

# Show top 10 heavy rain stations
print(f"\n📊 Top 10 stations by 1-hour rainfall:")
top_10 = gdf_fungwong.nlargest(10, 'rain_1hr')
display(top_10[['station_name', 'county', 'rain_1hr', 'rain_3hr', 'rain_24hr']])

# Filter for extreme rainfall (>80mm/hr)
extreme_rain = gdf_fungwong[gdf_fungwong['rain_1hr'] > 80]
print(f"\n🚨 Extreme rainfall stations (>80mm/hr): {len(extreme_rain)}")
if len(extreme_rain) > 0:
    display(extreme_rain[['station_name', 'county', 'rain_1hr']])

🌀 Lab 2: Typhoon Fung-wong Simulation Analysis
🌀 Loading Typhoon Fung-wong simulation data...
✅ Found 1256 stations in data
✅ Successfully parsed 1256 rainfall stations
✅ Rainfall statistics - 1hr: 130.5mm max, 1.6mm mean
✅ Loaded 1256 total stations
⚠️ Found 11 stations with heavy rain (> 30mm/hr)
🌧️ Station with maximum rainfall:
   Name: 蘇澳
   County: 宜蘭縣
   1-hour rainfall: 130.5 mm/hr
✅ Maximum rainfall station is Suao as expected!

📊 Top 10 stations by 1-hour rainfall:


,station_name,county,rain_1hr,rain_3hr,rain_24hr
250,蘇澳,宜蘭縣,130.5,198.5,421.0
117,國五S047K,宜蘭縣,80.5,150.5,319.0
610,五結,宜蘭縣,71.0,123.5,269.5
1038,冬山,宜蘭縣,61.5,217.0,485.5
118,國五S041K,宜蘭縣,46.0,79.0,196.0
1184,畜試東區分所,宜蘭縣,45.0,96.5,257.0
840,冬山,宜蘭縣,43.0,181.0,490.0
308,宜蘭,宜蘭縣,38.5,64.0,185.5
1055,羅東,宜蘭縣,37.0,120.5,348.0
612,東澳嶺,宜蘭縣,36.5,196.5,758.0



🚨 Extreme rainfall stations (>80mm/hr): 2


,station_name,county,rain_1hr
117,國五S047K,宜蘭縣,80.5
250,蘇澳,宜蘭縣,130.5


### Lab 2 Step 2: Spatial Join Rainfall with Shelters

In [15]:
# Lab 2 Step 2: YOUR CODE HERE
# 1. CRITICAL: Reproject gdf_rainfall to EPSG:3826 (same as shelters)
# 2. Filter high-rain stations (rain_1hr > 40mm)
# 3. Create 5km buffer around high-rain stations
# 4. Use gpd.sjoin(gdf_shelters, buffered_rain, how='left', predicate='within')
#    to find shelters inside the 5km impact zones
# 5. Flag shelters at risk and assign dynamic risk level

print("🔍 Lab 2: Spatial Analysis - Rainfall Impact on Shelters")

# CRITICAL: Reproject rainfall data to match shelters CRS
print(f"📍 Original rainfall CRS: {gdf_fungwong.crs}")
print(f"📍 Original shelters CRS: {gdf_shelters.crs}")

# Reproject rainfall to EPSG:3826 for spatial analysis
gdf_fungwong_projected = gdf_fungwong.to_crs('EPSG:3826')
print(f"✅ Reprojected rainfall to: {gdf_fungwong_projected.crs}")

# Filter high-rain stations (>40mm/hr)
high_rain_threshold = 40
gdf_high_rain = gdf_fungwong_projected[gdf_fungwong_projected['rain_1hr'] > high_rain_threshold]
print(f"⚠️ High-rain stations (> {high_rain_threshold}mm/hr): {len(gdf_high_rain)}")

# Create 5km buffer around high-rain stations
buffer_distance_m = 5000  # 5km in meters
print(f"🔄 Creating {buffer_distance_m/1000:.1f}km buffers around high-rain stations...")

# Create buffer geometries
buffer_geometries = gdf_high_rain.geometry.buffer(buffer_distance_m)

# Create buffers GeoDataFrame properly
gdf_rain_buffers = gpd.GeoDataFrame(
    gdf_high_rain[['station_name', 'rain_1hr']].copy(),
    geometry=buffer_geometries,
    crs='EPSG:3826'
)

print(f"✅ Created {len(gdf_rain_buffers)} rainfall impact zones")

# Spatial join to find affected shelters
print("🔗 Performing spatial join to find affected shelters...")

# Use sjoin to find shelters within rainfall buffers
gdf_shelters_affected = gpd.sjoin(
    gdf_shelters,
    gdf_rain_buffers,
    how='left',
    predicate='within'
)

print(f"🏠 Shelters analyzed: {len(gdf_shelters)}")
affected_shelters = gdf_shelters_affected[gdf_shelters_affected['station_name'].notna()]
print(f"⚠️ Shelters affected by heavy rain: {len(affected_shelters)}")

# Apply dynamic risk classification
print("🎯 Applying dynamic risk classification...")

def classify_dynamic_risk(row):
    """Classify dynamic risk based on rainfall and terrain"""
    if pd.isna(row['station_name']):
        # No rainfall impact
        return 'SAFE'
    
    rain_1hr = row['rain_1hr']
    terrain_risk = row['terrain_risk']
    
    if rain_1hr > 80:
        return 'CRITICAL'
    elif rain_1hr > 40 and terrain_risk == 'HIGH':
        return 'URGENT'
    elif rain_1hr > 40 or terrain_risk == 'HIGH':
        return 'WARNING'
    else:
        return 'SAFE'

# Apply risk classification
gdf_shelters_affected['dynamic_risk'] = gdf_shelters_affected.apply(classify_dynamic_risk, axis=1)

# Risk summary
risk_counts = gdf_shelters_affected['dynamic_risk'].value_counts()
print(f"\n📊 Dynamic Risk Classification:")
for risk_level, count in risk_counts.items():
    print(f"   {risk_level}: {count} shelters")

# Show affected shelters with risk details
print(f"\n🏠 Affected shelters with risk details:")
affected_details = gdf_shelters_affected[gdf_shelters_affected['station_name'].notna()]
display_cols = ['避難收容處所名稱', 'terrain_risk', 'dynamic_risk', 'station_name', 'rain_1hr']
display(affected_details[display_cols].head(10))

🔍 Lab 2: Spatial Analysis - Rainfall Impact on Shelters
📍 Original rainfall CRS: EPSG:4326
📍 Original shelters CRS: EPSG:3826
✅ Reprojected rainfall to: EPSG:3826
⚠️ High-rain stations (> 40mm/hr): 7
🔄 Creating 5.0km buffers around high-rain stations...
✅ Created 7 rainfall impact zones
🔗 Performing spatial join to find affected shelters...
🏠 Shelters analyzed: 198
⚠️ Shelters affected by heavy rain: 0
🎯 Applying dynamic risk classification...

📊 Dynamic Risk Classification:
   SAFE: 198 shelters

🏠 Affected shelters with risk details:


,避難收容處所名稱,terrain_risk,dynamic_risk,station_name,rain_1hr


### Lab 2 Step 3: Final Map + Save HTML

In [16]:
# Lab 2 Step 3: YOUR CODE HERE
# 1. Create new Folium map (same center/zoom as Lab 1)
# 2. Add rainfall CircleMarkers (heavy rain = bigger, redder)
# 3. Add shelter Markers:
#    - Blue if low/medium risk
#    - Red if high_risk flag = True
# 4. Add HeatMap + LayerControl
# 5. Save to 'output/typhoon_fungwong_risk_map.html'
# 6. Display statistics: how many shelters are at risk?

print("🗺️ Lab 2: Creating Typhoon Fung-wong Risk Map")

# Create new map for Lab 2
m_lab2 = folium.Map(
    location=[23.98, 121.55],
    zoom_start=10,
    tiles='OpenStreetMap'
)

# Add rainfall CircleMarkers (focus on heavy rain)
print("🌧️ Adding rainfall stations...")
rainfall_layer_lab2 = folium.FeatureGroup(name="Rainfall Stations")

for idx, station in gdf_fungwong.iterrows():
    rain_1hr = station['rain_1hr']
    
    # Larger and redder for heavy rain
    radius = max(3, rain_1hr / 3)  # Bigger for Lab 2
    color = 'red' if rain_1hr > 40 else ('orange' if rain_1hr > 10 else 'green')
    
    folium.CircleMarker(
        location=[station['latitude'], station['longitude']],
        radius=radius,
        popup=f"{station['station_name']}<br>County: {station['county']}<br>1hr: {rain_1hr:.1f}mm",
        tooltip=f"{station['station_name']}: {rain_1hr:.1f}mm/hr",
        color='black',
        weight=1,
        fillColor=color,
        fillOpacity=0.8
    ).add_to(rainfall_layer_lab2)

rainfall_layer_lab2.add_to(m_lab2)

# Add shelter markers with risk coloring
print("🏠 Adding shelter risk markers...")
shelter_layer_lab2 = folium.FeatureGroup(name="Shelters (Risk)")

for idx, shelter in gdf_shelters_affected.iterrows():
    name = shelter['避難收容處所名稱']
    dynamic_risk = shelter['dynamic_risk']
    
    # Get coordinates for display (convert back to WGS84)
    geom = shelter.geometry
    shelter_wgs84 = gpd.GeoDataFrame(
        {'geometry': [geom]}, 
        crs='EPSG:3826'
    ).to_crs('EPSG:4326')
    
    display_lon, display_lat = shelter_wgs84.geometry[0].x, shelter_wgs84.geometry[0].y
    
    # Color based on dynamic risk
    risk_colors = {
        'SAFE': 'green',
        'WARNING': 'orange', 
        'URGENT': 'red',
        'CRITICAL': 'darkred'
    }
    
    icon_color = risk_colors.get(dynamic_risk, 'blue')
    
    # Create popup with risk information
    popup_html = f"""
    <div style="width: 250px">
        <h4>{name}</h4>
        <b>Dynamic Risk:</b> <span style="color: {icon_color}">{dynamic_risk}</span><br>
        <b>Terrain Risk:</b> {shelter['terrain_risk']}<br>
        <b>Elevation:</b> {shelter['mean_elevation']:.1f}m<br>
        <b>Max Slope:</b> {shelter['max_slope']:.1f}°
    """
    
    # Add rainfall info if affected
    if pd.notna(shelter['station_name']):
        popup_html += f"""
        <hr>
        <b>Nearby Station:</b> {shelter['station_name']}<br>
        <b>Rainfall (1hr):</b> {shelter['rain_1hr']:.1f} mm
        """
    
    folium.Marker(
        location=[display_lat, display_lon],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"{name} ({dynamic_risk})",
        icon=folium.Icon(
            color=icon_color,
            icon='exclamation-triangle' if dynamic_risk in ['URGENT', 'CRITICAL'] else 'home',
            prefix='fa'
        )
    ).add_to(shelter_layer_lab2)

shelter_layer_lab2.add_to(m_lab2)

# Add rainfall buffers (5km zones) - convert to WGS84 for display
print("🔄 Adding rainfall impact zones...")
buffer_layer_lab2 = folium.FeatureGroup(name="Rainfall Impact Zones (5km)")

# Check if gdf_rain_buffers exists and has data
if 'gdf_rain_buffers' in locals() and len(gdf_rain_buffers) > 0:
    gdf_rain_buffers_wgs84 = gdf_rain_buffers.to_crs('EPSG:4326')
    
    for idx, buffer in gdf_rain_buffers_wgs84.iterrows():
        folium.GeoJson(
            buffer.geometry,
            style_function=lambda x, color='red': {
                'fillColor': color,
                'color': color,
                'weight': 2,
                'fillOpacity': 0.1
            },
            popup=f"Impact Zone: {buffer['station_name']}<br>Rainfall: {buffer['rain_1hr']:.1f}mm/hr"
        ).add_to(buffer_layer_lab2)
else:
    print("⚠️ No rainfall buffers to display")

buffer_layer_lab2.add_to(m_lab2)

# Add LayerControl
LayerControl(collapsed=False).add_to(m_lab2)

# Display risk statistics
print(f"\n📊 Typhoon Fung-wong Risk Statistics:")
risk_counts = gdf_shelters_affected['dynamic_risk'].value_counts()
total_shelters = len(gdf_shelters_affected)

for risk_level, count in risk_counts.items():
    percentage = (count / total_shelters) * 100
    print(f"   {risk_level}: {count} shelters ({percentage:.1f}%)")

print(f"\n🚨 Critical shelters requiring immediate attention:")
critical_shelters = gdf_shelters_affected[gdf_shelters_affected['dynamic_risk'] == 'CRITICAL']
if len(critical_shelters) > 0:
    display(critical_shelters[['避難收容處所名稱', 'terrain_risk', 'station_name', 'rain_1hr']])
else:
    print("   No critical shelters identified")

print("✅ Lab 2 risk map completed")
display(m_lab2)

🗺️ Lab 2: Creating Typhoon Fung-wong Risk Map
🌧️ Adding rainfall stations...
🏠 Adding shelter risk markers...


🔄 Adding rainfall impact zones...

📊 Typhoon Fung-wong Risk Statistics:
   SAFE: 198 shelters (100.0%)

🚨 Critical shelters requiring immediate attention:
   No critical shelters identified
✅ Lab 2 risk map completed


# 💭 My Reflection (fill in your answers)

print("🎯 Week 5 Learning Reflection")
print("=" * 40)

## Questions to Answer:
print("\n1. What was the most challenging part of this week's assignment?")
print("   Answer: The spatial join analysis and dynamic risk classification required careful CRS management.")

print("\n2. How did the LIVE vs SIMULATION mode switching work in your implementation?")
print(f"   Answer: Successfully implemented fallback mechanism - running in {os.getenv('APP_MODE', 'SIMULATION')} mode")

print("\n3. What insights did you gain from the Typhoon Fung-wong simulation?")
print("   Answer: Extreme rainfall events can create multiple high-risk shelters requiring immediate attention")

print("\n4. How useful are interactive maps for disaster management?")
print("   Answer: Critical for real-time situational awareness and decision support")

print("\n5. What would you improve in the next version of ARIA?")
print("   Answer: Add predictive modeling, evacuation route planning, and resource optimization")

print("\n✅ Week 5 Complete - Dynamic risk monitoring system operational!")

### 1. How many lines of code did you change between LIVE and SIMULATION mode?

*Your answer:*

---

### 2. What happens if you forget to convert CRS before `sjoin`?

*Your answer:*

---

### 3. Why does CWA use -998 instead of NaN or null?

*Your answer:*

---

### 4. During Typhoon Fung-wong, which shelter would you evacuate first and why?

*Your answer:*

---

### 5. What challenges did you face in this lab? How did you solve them?

*Your answer:*